In [3]:
import chess.pgn
import chess
import numpy as np

In [ ]:
with open("../data/chess/lichess_elite_2020-06.pgn") as f:
    game = chess.pgn.read_game(f)       # reads the next game each call; None at end of file

In [ ]:
board = game.board()
for move in game.mainline_moves():
    print(board.turn, move)             # turn: True = white to move
    board.push(move)


In [6]:
# Convert board to tensor
#######################################################
def board_to_tensor(board):
    '''
    Converts a python-chess object (which itself is a conversion from lichess PNGs)
    into a tensor
    '''
    # Create zeroed tensor
    board_tensor = np.zeros((12, 8, 8), dtype=np.uint8)
    # Flip board if black so that all boards are from the 'white' pov
    if board.turn == chess.BLACK:
        board = board.mirror()
    # Loop over piecemap 
    for square, piece in board.piece_map().items():
        # Calculate plane
        # Plane is piece type 0-5 for self, 6-11 for opponent
        piece_type = piece.piece_type - 1
        plane = piece_type
        if piece.color == chess.BLACK:
            plane += 6
        # Calculate position
        row = chess.square_rank(square)
        col = chess.square_file(square)
        # Set the piece plane, x, y to 1 (representing a piece in those coords)
        board_tensor[plane, row, col] = 1

    return board_tensor

In [1]:
# Encoding
all_moves = []
# Loop through all from_square -> to square combinations
for from_sq in range(64):
    for to_sq in range(64):
        # Skip when they are equal
        if from_sq == to_sq:
            continue
        from_row = from_sq // 8
        from_col = from_sq % 8
        to_row = to_sq // 8
        to_col = to_sq % 8
        dr = abs(to_row - from_row)
        dc = abs(to_col - from_col)

        if dr == 0 or dc == 0 or dr == dc or (dr, dc) == (1, 2) or (dr, dc) == (2, 1):
            all_moves.append((from_sq, to_sq))

print(len(all_moves))

1792


In [4]:
# Lookup: (from_sq, to_sq) -> index
move_to_idx = {pair: i for i, pair in enumerate(all_moves)}

def move_to_index(move, board):
    '''
    Converts a python-chess move into an index 0-1791.
    board is the position before the move; moves are mirrored if black to move,
    to match board_to_tensor
    '''
    from_sq, to_sq = move.from_square, move.to_square
    if board.turn == chess.BLACK:
        from_sq = chess.square_mirror(from_sq)
        to_sq = chess.square_mirror(to_sq)
    return move_to_idx[(from_sq, to_sq)]

def index_to_move(idx, board):
    '''
    Converts an index 0-1791 back into a python-chess move for this board.
    Pawn moves to the last rank are always queen promotions
    '''
    from_sq, to_sq = all_moves[idx]
    if board.turn == chess.BLACK:
        from_sq = chess.square_mirror(from_sq)
        to_sq = chess.square_mirror(to_sq)
    promotion = None
    if board.piece_type_at(from_sq) == chess.PAWN and chess.square_rank(to_sq) in (0, 7):
        promotion = chess.QUEEN
    return chess.Move(from_sq, to_sq, promotion=promotion)


In [7]:
import os

PGN_PATH = "../data/chess/lichess_elite_2020-06.pgn"
OUT_DIR = "../data/chess/processed"
MAX_GAMES = 12_000          # ~1M positions

boards, moves, game_ids = [], [], []
skipped = 0

with open(PGN_PATH) as f:
    for game_id in range(MAX_GAMES):
        game = chess.pgn.read_game(f)
        if game is None:    # end of file
            break

        board = game.board()
        for move in game.mainline_moves():
            # Skip underpromotions (can't be encoded, always promote to queen)
            if move.promotion is None or move.promotion == chess.QUEEN:
                boards.append(board_to_tensor(board))
                moves.append(move_to_index(move, board))
                game_ids.append(game_id)
            else:
                skipped += 1
            board.push(move)

        if game_id % 1000 == 0:
            print(f"game {game_id}, positions {len(boards)}")

# Lists -> arrays
boards = np.stack(boards)                       # (N, 12, 8, 8) uint8
moves = np.array(moves, dtype=np.int16)         # (N,)
game_ids = np.array(game_ids, dtype=np.int32)   # (N,) for splitting train/val by game

os.makedirs(OUT_DIR, exist_ok=True)
np.save(f"{OUT_DIR}/boards.npy", boards)
np.save(f"{OUT_DIR}/moves.npy", moves)
np.save(f"{OUT_DIR}/game_ids.npy", game_ids)

print(boards.shape, moves.shape, f"skipped {skipped} underpromotions")


game 0, positions 72
game 1000, positions 82820
game 2000, positions 166347
game 3000, positions 246656
game 4000, positions 328891
game 5000, positions 411185
game 6000, positions 493359
game 7000, positions 576493
game 8000, positions 660076
game 9000, positions 743739
game 10000, positions 827641
game 11000, positions 910633
(993922, 12, 8, 8) (993922,) skipped 21 underpromotions
